<a href="https://colab.research.google.com/github/AidoruFusion/northstar-databases-analytics/blob/main/Database_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/AidoruFusion/northstar-databases-analytics.git

fatal: destination path 'northstar-databases-analytics' already exists and is not an empty directory.


# Python Library Imports

This step loads the Python tools needed to:
- read CSV files;
- work with folders;
- clean missing and duplicate values;
- save cleaned datasets.

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
from pathlib import Path

# Uploading of CSV Files
The raw CSV files are uploaded into the Colab working folder.

The cleaned files will be saved into separate folders:
- one folder for cleaned CSV files;
- one folder for JSON files.

In [ ]:
from pathlib import Path

repo_folder = Path("/content/northstar-databases-analytics")

raw_folder = Path("/content")
cleaned_csv_folder = repo_folder / "data/cleaned/csv"
json_folder = repo_folder / "data/cleaned/json"

cleaned_csv_folder.mkdir(parents=True, exist_ok=True)
json_folder.mkdir(parents=True, exist_ok=True)

print("Raw folder:", raw_folder)
print("Cleaned CSV folder:", cleaned_csv_folder)
print("JSON folder:", json_folder)

Raw folder: /content
Cleaned CSV folder: /content/northstar-databases-analytics/data/cleaned/csv
JSON folder: /content/northstar-databases-analytics/data/cleaned/json


# Checking Available CSV Files

This step checks that the raw CSV files are available before cleaning starts.

This is important because each file represents a different part of NorthStar's operations.

In [ ]:
csv_files = [
    "customers.csv",
    "orders.csv",
    "deliveries.csv",
    "drivers.csv",
    "vehicles.csv",
    "hubs.csv",
    "complaints.csv",
    "incidents.csv",
    "app_events.csv",
    "data_dictionary.csv"
]

for file_name in csv_files:
    file_path = raw_folder / file_name
    print(file_name, "exists:", file_path.exists())

customers.csv exists: True
orders.csv exists: True
deliveries.csv exists: True
drivers.csv exists: True
vehicles.csv exists: True
hubs.csv exists: True
complaints.csv exists: True
incidents.csv exists: True
app_events.csv exists: True
data_dictionary.csv exists: True


# Find what needs cleaning

Before cleaning, each file is inspected.

The checks look for:
- missing values;
- duplicate rows;
- data types;
- inconsistent zone names.

This helps explain why cleaning is needed instead of cleaning blindly.

In [ ]:
for file_name in csv_files:
    df = pd.read_csv(raw_folder / file_name)

    print("\n" + "=" * 70)
    print(file_name)
    print("Rows and columns:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())

    print("\nMissing values:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

    zone_columns = [col for col in df.columns if "zone" in col.lower()]
    for col in zone_columns:
        print(f"\nUnique values in {col}:")
        print(sorted(df[col].dropna().astype(str).unique()))


customers.csv
Rows and columns: (650, 9)
Duplicate rows: 0

Missing values:
loyalty_score        20
preferred_channel    13
dtype: int64

Unique values in home_zone:
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']

orders.csv
Rows and columns: (1250, 11)
Duplicate rows: 0

Missing values:
booking_channel    25
dtype: int64

Unique values in pickup_zone:
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']

Unique values in dropoff_zone:
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']

deliveries.csv
Rows and columns: (950, 13)
Duplicate rows: 0

Missing values:
delivery_completed_at            19
customer_rating_post_delivery    14
dtype: int64

drivers.csv
Rows and columns:

# Cleaning issues identified

The audit shows that the main cleaning issues are:

- zone names are inconsistent, for example `Central`, `CENTRAL`, and `Ctr`;
- some date columns are stored as text;
- some numeric fields need conversion before analysis;
- some files contain missing values;
- delivery duration needs to be calculated from dispatch and completion times;
- invalid values need checking, such as negative distances or ratings outside the valid range.

Cleaning is needed because NorthStar's analysis depends on comparing zones, customers, orders, deliveries, vehicles, drivers, complaints, incidents, and app activity correctly.

# Create helper cleaning functions

This section stores reusable cleaning functions.

Using functions keeps the notebook organised and avoids repeating the same cleaning code for every file.

In [ ]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df


def clean_text_columns(df):
    df = df.copy()
    text_columns = df.select_dtypes(include="object").columns

    for col in text_columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({
            "nan": np.nan,
            "None": np.nan,
            "": np.nan
        })

    return df


def basic_cleaning(df):
    df = clean_column_names(df)
    df = df.drop_duplicates()
    df = clean_text_columns(df)
    return df


def convert_to_numeric(df, columns):
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def convert_to_datetime(df, columns):
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def save_cleaned_file(df, file_name):
    output_path = cleaned_csv_folder / file_name
    df.to_csv(output_path, index=False)
    print(file_name, "saved successfully.")

# Cleaning of Zones
For example:
- `Central`, `CENTRAL`, and `Ctr` mean the same zone;
- `North`, `NORTH`, and `north` mean the same zone;
- `RiverSide` and `Riverside` mean the same zone.

This needs cleaning because NorthStar's case study focuses on comparing performance across city zones. If the names are not standardised, one real zone may be counted as multiple different zones.


In [ ]:
zone_mapping = {
    "CENTRAL": "Central",
    "Central": "Central",
    "Ctr": "Central",
    "ctr": "Central",

    "NORTH": "North",
    "North": "North",
    "north": "North",

    "SOUTH": "South",
    "South": "South",
    "south": "South",

    "EAST": "East",
    "East": "East",
    "east": "East",

    "WEST": "West",
    "West": "West",
    "west": "West",

    "AIRPORT": "Airport",
    "Airport": "Airport",
    "airport": "Airport",

    "RiverSide": "Riverside",
    "Riverside": "Riverside",
    "riverside": "Riverside"
}

# Cleaning customers.csv

The customers file contains customer information such as age, home zone, signup date, loyalty score, app engagement score, preferred channel, and account status.

This file needs cleaning because:
- `home_zone` may contain inconsistent zone names such as `Central`, `CENTRAL`, and `Ctr`;
- `signup_date` should be converted into date format;
- age and scores should be numeric;
- unrealistic age values should be treated as missing.

In [ ]:
customers_raw = pd.read_csv("/content/customers.csv")
customers = customers_raw.copy()

display(customers.head())
print(customers.info())

,customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status
0,C0001,26,North,SME,2024-11-27 04:25:00,44.9,69.2,App,Active
1,C0002,61,AIRPORT,Consumer,2025-10-28 01:04:00,55.4,66.6,App,Active
2,C0003,66,East,Consumer,2025-07-02 03:23:00,75.9,33.8,NaN,Active
3,C0004,75,CENTRAL,Consumer,2025-08-19 01:58:00,32.5,33.0,App,Active
4,C0005,26,Riverside,Consumer,2025-06-03 06:02:00,55.9,100.0,Web,Active


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           650 non-null    object 
 1   age                   650 non-null    int64  
 2   home_zone             650 non-null    object 
 3   customer_type         650 non-null    object 
 4   signup_date           650 non-null    object 
 5   loyalty_score         630 non-null    float64
 6   app_engagement_score  650 non-null    float64
 7   preferred_channel     637 non-null    object 
 8   account_status        650 non-null    object 
dtypes: float64(2), int64(1), object(6)
memory usage: 45.8+ KB
None


In [ ]:
customers = basic_cleaning(customers)

customers["home_zone"] = customers["home_zone"].replace(zone_mapping)

customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
customers["age"] = pd.to_numeric(customers["age"], errors="coerce")
customers["loyalty_score"] = pd.to_numeric(customers["loyalty_score"], errors="coerce")
customers["app_engagement_score"] = pd.to_numeric(customers["app_engagement_score"], errors="coerce")

customers.loc[(customers["age"] < 16) | (customers["age"] > 100), "age"] = np.nan

display(customers.head())

,customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status
0,C0001,26.0,North,SME,2024-11-27 04:25:00,44.9,69.2,App,Active
1,C0002,61.0,Airport,Consumer,2025-10-28 01:04:00,55.4,66.6,App,Active
2,C0003,66.0,East,Consumer,2025-07-02 03:23:00,75.9,33.8,NaN,Active
3,C0004,75.0,Central,Consumer,2025-08-19 01:58:00,32.5,33.0,App,Active
4,C0005,26.0,Riverside,Consumer,2025-06-03 06:02:00,55.9,100.0,Web,Active


In [ ]:
save_cleaned_file(customers, "customers_cleaned.csv")

customers_cleaned.csv saved successfully.


# Cleaning orders.csv

The orders file records customer service bookings.

This file needs cleaning because:
- pickup and drop-off zones may contain inconsistent values such as `Airport` and `AIRPORT`;
- order dates need to be converted into datetime format;
- order value and promised window should be numeric;
- negative order values are not valid for analysis.

In [ ]:
orders_raw = pd.read_csv("/content/orders.csv")
orders = orders_raw.copy()

display(orders.head())
print(orders.info())

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
0,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
1,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
2,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
3,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
4,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1250 entries, 0 to 1249
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               1250 non-null   object 
 1   customer_id            1250 non-null   object 
 2   service_type           1250 non-null   object 
 3   order_created_at       1250 non-null   object 
 4   promised_window_hours  1250 non-null   int64  
 5   pickup_zone            1250 non-null   object 
 6   dropoff_zone           1250 non-null   object 
 7   priority_level         1250 non-null   object 
 8   order_value            1250 non-null   float64
 9   booking_channel        1225 non-null   object 
 10  special_handling_flag  1250 non-null   int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 107.6+ KB
None


In [ ]:
orders = basic_cleaning(orders)

orders["pickup_zone"] = orders["pickup_zone"].replace(zone_mapping)
orders["dropoff_zone"] = orders["dropoff_zone"].replace(zone_mapping)

orders["order_created_at"] = pd.to_datetime(orders["order_created_at"], errors="coerce")
orders["promised_window_hours"] = pd.to_numeric(orders["promised_window_hours"], errors="coerce")
orders["order_value"] = pd.to_numeric(orders["order_value"], errors="coerce")
orders["special_handling_flag"] = pd.to_numeric(orders["special_handling_flag"], errors="coerce")

orders.loc[orders["order_value"] < 0, "order_value"] = np.nan

display(orders.head())

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
0,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
1,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,Airport,Low,109.30,App,0
2,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,Airport,High,33.50,Phone,0
3,O00004,C0520,Parcel,2025-01-11 17:15:00,2,Riverside,North,Medium,10.04,App,1
4,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,South,Low,125.58,Phone,0


In [ ]:
save_cleaned_file(orders, "orders_cleaned.csv")

orders_cleaned.csv saved successfully.


# Cleaning deliveries.csv

The deliveries file records dispatch, delivery completion, route distance, route overrides, ratings, and fuel or charging cost.

This file needs cleaning because:
- dispatch and completion times need datetime conversion;
- numeric delivery fields need to be converted properly;
- ratings should be between 1 and 5;
- route distance should not be negative;
- delivery duration can be calculated for later delay analysis.

In [ ]:
deliveries_raw = pd.read_csv("/content/deliveries.csv")
deliveries = deliveries_raw.copy()

display(deliveries.head())
print(deliveries.info())

,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
0,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
1,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
2,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51
3,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62
4,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   delivery_id                    950 non-null    object 
 1   order_id                       950 non-null    object 
 2   driver_id                      950 non-null    object 
 3   vehicle_id                     950 non-null    object 
 4   hub_id                         950 non-null    object 
 5   dispatch_time                  950 non-null    object 
 6   delivery_completed_at          931 non-null    object 
 7   delivery_status                950 non-null    object 
 8   route_distance_km              950 non-null    float64
 9   manual_route_override_count    950 non-null    int64  
 10  proof_of_completion_missing    950 non-null    int64  
 11  customer_rating_post_delivery  936 non-null    float64
 12  fuel_or_charge_cost            950 non-null    flo

In [ ]:
deliveries = basic_cleaning(deliveries)

deliveries["dispatch_time"] = pd.to_datetime(deliveries["dispatch_time"], errors="coerce")
deliveries["delivery_completed_at"] = pd.to_datetime(deliveries["delivery_completed_at"], errors="coerce")

numeric_cols = [
    "route_distance_km",
    "manual_route_override_count",
    "proof_of_completion_missing",
    "customer_rating_post_delivery",
    "fuel_or_charge_cost"
]

for col in numeric_cols:
    deliveries[col] = pd.to_numeric(deliveries[col], errors="coerce")

deliveries["delivery_duration_hours"] = (
    deliveries["delivery_completed_at"] - deliveries["dispatch_time"]
).dt.total_seconds() / 3600

deliveries.loc[deliveries["route_distance_km"] < 0, "route_distance_km"] = np.nan

deliveries.loc[
    (deliveries["customer_rating_post_delivery"] < 1) |
    (deliveries["customer_rating_post_delivery"] > 5),
    "customer_rating_post_delivery"
] = np.nan

deliveries.loc[deliveries["delivery_duration_hours"] < 0, "delivery_duration_hours"] = np.nan

display(deliveries.head())

,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost,delivery_duration_hours
0,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05,22.149973
1,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41,NaN
2,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51,1.108991
3,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62,23.985584
4,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22,4.042814


In [ ]:
save_cleaned_file(deliveries, "deliveries_cleaned.csv")

deliveries_cleaned.csv saved successfully.


# Cleaning drivers.csv

The drivers file contains workforce and driver performance data.

This file needs cleaning because:
- `base_zone` may contain inconsistent zone names such as `NORTH`, `North`, and `north`;
- training score and driver rating need to be numeric;
- ratings should stay within a valid range;
- training scores should stay between 0 and 100.



In [ ]:
drivers_raw = pd.read_csv("/content/drivers.csv")
drivers = drivers_raw.copy()

display(drivers.head())
print(drivers.info())

,driver_id,base_zone,employment_type,years_experience,training_score,driver_rating,shift_preference,active_flag
0,D001,AIRPORT,FullTime,8,67.8,3.54,Morning,1
1,D002,Central,FullTime,4,42.4,3.94,Evening,1
2,D003,AIRPORT,FullTime,11,96.5,5.00,Evening,1
3,D004,Airport,PartTime,13,88.9,4.75,Morning,1
4,D005,north,FullTime,3,69.7,4.14,Morning,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   driver_id         170 non-null    object 
 1   base_zone         170 non-null    object 
 2   employment_type   170 non-null    object 
 3   years_experience  170 non-null    int64  
 4   training_score    163 non-null    float64
 5   driver_rating     170 non-null    float64
 6   shift_preference  170 non-null    object 
 7   active_flag       170 non-null    int64  
dtypes: float64(2), int64(2), object(4)
memory usage: 10.8+ KB
None


In [ ]:
drivers = basic_cleaning(drivers)

drivers["base_zone"] = drivers["base_zone"].replace(zone_mapping)

drivers["years_experience"] = pd.to_numeric(drivers["years_experience"], errors="coerce")
drivers["training_score"] = pd.to_numeric(drivers["training_score"], errors="coerce")
drivers["driver_rating"] = pd.to_numeric(drivers["driver_rating"], errors="coerce")
drivers["active_flag"] = pd.to_numeric(drivers["active_flag"], errors="coerce")

drivers.loc[(drivers["training_score"] < 0) | (drivers["training_score"] > 100), "training_score"] = np.nan
drivers.loc[(drivers["driver_rating"] < 1) | (drivers["driver_rating"] > 5), "driver_rating"] = np.nan

display(drivers.head())

,driver_id,base_zone,employment_type,years_experience,training_score,driver_rating,shift_preference,active_flag
0,D001,Airport,FullTime,8,67.8,3.54,Morning,1
1,D002,Central,FullTime,4,42.4,3.94,Evening,1
2,D003,Airport,FullTime,11,96.5,5.00,Evening,1
3,D004,Airport,PartTime,13,88.9,4.75,Morning,1
4,D005,North,FullTime,3,69.7,4.14,Morning,1


In [ ]:
save_cleaned_file(drivers, "drivers_cleaned.csv")

drivers_cleaned.csv saved successfully.


# Cleaning vehicles.csv

The vehicles file contains fleet and asset condition data.

This file needs cleaning because:
- assigned zones may be written inconsistently;
- commission date needs datetime conversion;
- battery health and odometer readings need to be numeric;
- battery health must be between 0 and 100;
- odometer values should not be negative.

In [ ]:
vehicles_raw = pd.read_csv("/content/vehicles.csv")
vehicles = vehicles_raw.copy()

display(vehicles.head())
print(vehicles.info())

,vehicle_id,vehicle_type,assigned_zone,commission_date,battery_health_pct,odometer_km,maintenance_status,telematics_version
0,V001,EV,North,2024-12-28 23:48:00,71.8,56928,Active,v2.2
1,V002,EV,AIRPORT,2024-04-21 16:14:00,67.9,159368,InRepair,v2.2
2,V003,CargoVan,north,2025-11-24 23:59:00,91.7,219359,Active,v2.1
3,V004,Hybrid,RiverSide,2024-06-07 13:21:00,NaN,36310,Active,v2.2
4,V005,CargoVan,West,2025-11-15 11:08:00,58.6,146638,Active,v2.2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   vehicle_id          120 non-null    object 
 1   vehicle_type        120 non-null    object 
 2   assigned_zone       120 non-null    object 
 3   commission_date     120 non-null    object 
 4   battery_health_pct  116 non-null    float64
 5   odometer_km         120 non-null    int64  
 6   maintenance_status  120 non-null    object 
 7   telematics_version  120 non-null    object 
dtypes: float64(1), int64(1), object(6)
memory usage: 7.6+ KB
None


In [ ]:
vehicles = basic_cleaning(vehicles)

vehicles["assigned_zone"] = vehicles["assigned_zone"].replace(zone_mapping)

vehicles["commission_date"] = pd.to_datetime(vehicles["commission_date"], errors="coerce")
vehicles["battery_health_pct"] = pd.to_numeric(vehicles["battery_health_pct"], errors="coerce")
vehicles["odometer_km"] = pd.to_numeric(vehicles["odometer_km"], errors="coerce")

vehicles.loc[(vehicles["battery_health_pct"] < 0) | (vehicles["battery_health_pct"] > 100), "battery_health_pct"] = np.nan
vehicles.loc[vehicles["odometer_km"] < 0, "odometer_km"] = np.nan

display(vehicles.head())

,vehicle_id,vehicle_type,assigned_zone,commission_date,battery_health_pct,odometer_km,maintenance_status,telematics_version
0,V001,EV,North,2024-12-28 23:48:00,71.8,56928.0,Active,v2.2
1,V002,EV,Airport,2024-04-21 16:14:00,67.9,159368.0,InRepair,v2.2
2,V003,CargoVan,North,2025-11-24 23:59:00,91.7,219359.0,Active,v2.1
3,V004,Hybrid,Riverside,2024-06-07 13:21:00,NaN,36310.0,Active,v2.2
4,V005,CargoVan,West,2025-11-15 11:08:00,58.6,146638.0,Active,v2.2


In [ ]:
save_cleaned_file(vehicles, "vehicles_cleaned.csv")

vehicles_cleaned.csv saved successfully.


# Cleaning hubs.csv

The hubs file contains NorthStar's operational hub information.

This file needs cleaning because:
- hub zones should use consistent names;
- capacity score should be numeric;
- capacity score should stay between 0 and 100;
- hub data will support later zone and performance analysis.



In [ ]:
hubs_raw = pd.read_csv("/content/hubs.csv")
hubs = hubs_raw.copy()

display(hubs.head())
print(hubs.info())

,hub_id,hub_name,zone,hub_type,capacity_score
0,H01,North Exchange,North,Dispatch,82
1,H02,South Link,South,Dispatch,78
2,H03,East Dock,East,Warehouse,74
3,H04,West Gate,West,Dispatch,69
4,H05,Central Core,Central,Control,88


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   hub_id          8 non-null      object
 1   hub_name        8 non-null      object
 2   zone            8 non-null      object
 3   hub_type        8 non-null      object
 4   capacity_score  8 non-null      int64 
dtypes: int64(1), object(4)
memory usage: 452.0+ bytes
None


In [ ]:
hubs = basic_cleaning(hubs)

hubs["zone"] = hubs["zone"].replace(zone_mapping)

hubs["capacity_score"] = pd.to_numeric(hubs["capacity_score"], errors="coerce")
hubs.loc[(hubs["capacity_score"] < 0) | (hubs["capacity_score"] > 100), "capacity_score"] = np.nan

display(hubs.head())

,hub_id,hub_name,zone,hub_type,capacity_score
0,H01,North Exchange,North,Dispatch,82.0
1,H02,South Link,South,Dispatch,78.0
2,H03,East Dock,East,Warehouse,74.0
3,H04,West Gate,West,Dispatch,69.0
4,H05,Central Core,Central,Control,88.0


In [ ]:
save_cleaned_file(hubs, "hubs_cleaned.csv")

hubs_cleaned.csv saved successfully.


# Cleaning complaints.csv

The complaints file records customer service problems.

This file needs cleaning because:
- complaint dates need datetime conversion;
- compensation and resolution days should be numeric;
- negative compensation or resolution values are invalid;
- consistent complaint categories help later analysis.

In [ ]:
complaints_raw = pd.read_csv("/content/complaints.csv")
complaints = complaints_raw.copy()

display(complaints.head())
print(complaints.info())

,complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
0,CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11,23.99
1,CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4,21.64
2,CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16,26.41
3,CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7,23.44
4,CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1,16.18


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   complaint_id         320 non-null    object 
 1   customer_id          320 non-null    object 
 2   order_id             320 non-null    object 
 3   complaint_type       320 non-null    object 
 4   channel              320 non-null    object 
 5   severity             320 non-null    object 
 6   created_at           320 non-null    object 
 7   status               320 non-null    object 
 8   resolution_days      320 non-null    int64  
 9   compensation_amount  304 non-null    float64
dtypes: float64(1), int64(1), object(8)
memory usage: 25.1+ KB
None


In [ ]:
complaints = basic_cleaning(complaints)

complaints["created_at"] = pd.to_datetime(complaints["created_at"], errors="coerce")
complaints["resolution_days"] = pd.to_numeric(complaints["resolution_days"], errors="coerce")
complaints["compensation_amount"] = pd.to_numeric(complaints["compensation_amount"], errors="coerce")

complaints.loc[complaints["resolution_days"] < 0, "resolution_days"] = np.nan
complaints.loc[complaints["compensation_amount"] < 0, "compensation_amount"] = np.nan

display(complaints.head())

,complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
0,CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11.0,23.99
1,CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4.0,21.64
2,CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16.0,26.41
3,CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7.0,23.44
4,CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1.0,16.18


In [ ]:
save_cleaned_file(complaints, "complaints_cleaned.csv")

complaints_cleaned.csv saved successfully.


# Cleaning incidents.csv

The incidents file records operational problems linked to deliveries.

This file needs cleaning because:
- reported dates need datetime conversion;
- resolved hours should be numeric;
- negative resolution times are invalid;
- incident type and severity should be consistently formatted.

In [ ]:
incidents_raw = pd.read_csv("/content/incidents.csv")
incidents = incidents_raw.copy()

display(incidents.head())
print(incidents.info())

,incident_id,delivery_id,incident_type,reported_at,severity,resolution_status,resolved_hours
0,I0001,DL00221,BatteryAlert,2024-03-11 23:46:00,Medium,Escalated,12.3
1,I0002,DL00578,BatteryAlert,2024-02-21 10:56:00,Low,Open,9.6
2,I0003,DL00175,TemperatureIssue,2025-04-17 23:22:00,Medium,Open,22.0
3,I0004,DL00417,ProofMissing,2025-02-09 00:16:00,Medium,Closed,9.8
4,I0005,DL00897,RouteDeviation,2025-01-04 02:49:00,Low,Open,13.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   incident_id        280 non-null    object 
 1   delivery_id        280 non-null    object 
 2   incident_type      280 non-null    object 
 3   reported_at        280 non-null    object 
 4   severity           280 non-null    object 
 5   resolution_status  280 non-null    object 
 6   resolved_hours     263 non-null    float64
dtypes: float64(1), object(6)
memory usage: 15.4+ KB
None


In [ ]:
incidents = basic_cleaning(incidents)

incidents["reported_at"] = pd.to_datetime(incidents["reported_at"], errors="coerce")
incidents["resolved_hours"] = pd.to_numeric(incidents["resolved_hours"], errors="coerce")

incidents.loc[incidents["resolved_hours"] < 0, "resolved_hours"] = np.nan

display(incidents.head())

,incident_id,delivery_id,incident_type,reported_at,severity,resolution_status,resolved_hours
0,I0001,DL00221,BatteryAlert,2024-03-11 23:46:00,Medium,Escalated,12.3
1,I0002,DL00578,BatteryAlert,2024-02-21 10:56:00,Low,Open,9.6
2,I0003,DL00175,TemperatureIssue,2025-04-17 23:22:00,Medium,Open,22.0
3,I0004,DL00417,ProofMissing,2025-02-09 00:16:00,Medium,Closed,9.8
4,I0005,DL00897,RouteDeviation,2025-01-04 02:49:00,Low,Open,13.0


In [ ]:
save_cleaned_file(incidents, "incidents_cleaned.csv")

incidents_cleaned.csv saved successfully.


#Cleaning app_events.csv

The app events file records digital platform activity.

This file needs cleaning because:
- zone context may contain inconsistent values such as `Central`, `CENTRAL`, and `Ctr`;
- event timestamps need datetime conversion;
- API latency should be numeric;
- negative latency values are invalid;
- not every app event has an order ID, so missing order IDs should be kept.

In [ ]:
app_events_raw = pd.read_csv("/content/app_events.csv")
app_events = app_events_raw.copy()

display(app_events.head())
print(app_events.info())

,event_id,customer_id,order_id,event_timestamp,event_type,session_id,device_type,zone_context,api_latency_ms,success_flag
0,AE00001,C0488,NaN,2024-08-09 03:25:00,eta_refresh,S19847,Android,north,301,1
1,AE00002,C0595,O00950,2024-02-13 22:29:00,search_route,S32766,Android,SOUTH,60,1
2,AE00003,C0494,O00170,2025-08-11 09:29:00,chat_opened,S99516,iOS,Airport,1118,1
3,AE00004,C0407,O00756,2025-08-23 17:38:00,eta_refresh,S41236,iOS,CENTRAL,442,1
4,AE00005,C0506,NaN,2024-05-29 10:33:00,search_route,S12030,iOS,north,60,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   event_id         640 non-null    object
 1   customer_id      640 non-null    object
 2   order_id         496 non-null    object
 3   event_timestamp  640 non-null    object
 4   event_type       640 non-null    object
 5   session_id       640 non-null    object
 6   device_type      640 non-null    object
 7   zone_context     640 non-null    object
 8   api_latency_ms   640 non-null    int64 
 9   success_flag     640 non-null    int64 
dtypes: int64(2), object(8)
memory usage: 50.1+ KB
None


In [ ]:
app_events = basic_cleaning(app_events)

app_events["zone_context"] = app_events["zone_context"].replace(zone_mapping)

app_events["event_timestamp"] = pd.to_datetime(app_events["event_timestamp"], errors="coerce")
app_events["api_latency_ms"] = pd.to_numeric(app_events["api_latency_ms"], errors="coerce")
app_events["success_flag"] = pd.to_numeric(app_events["success_flag"], errors="coerce")

app_events.loc[app_events["api_latency_ms"] < 0, "api_latency_ms"] = np.nan

display(app_events.head())

,event_id,customer_id,order_id,event_timestamp,event_type,session_id,device_type,zone_context,api_latency_ms,success_flag
0,AE00001,C0488,NaN,2024-08-09 03:25:00,eta_refresh,S19847,Android,North,301.0,1
1,AE00002,C0595,O00950,2024-02-13 22:29:00,search_route,S32766,Android,South,60.0,1
2,AE00003,C0494,O00170,2025-08-11 09:29:00,chat_opened,S99516,iOS,Airport,1118.0,1
3,AE00004,C0407,O00756,2025-08-23 17:38:00,eta_refresh,S41236,iOS,Central,442.0,1
4,AE00005,C0506,NaN,2024-05-29 10:33:00,search_route,S12030,iOS,North,60.0,1


In [ ]:
save_cleaned_file(app_events, "app_events_cleaned.csv")

app_events_cleaned.csv saved successfully.


# Cleaning data_dictionary.csv

The data dictionary explains the meaning of the dataset files and fields.

This file needs light cleaning only because:
- it is mainly descriptive;
- text spacing should be cleaned;
- it supports documentation for the project.

In [ ]:
data_dictionary_raw = pd.read_csv("/content/data_dictionary.csv")
data_dictionary = data_dictionary_raw.copy()

display(data_dictionary.head())
print(data_dictionary.info())

,file_name,record_count,description
0,hubs.csv,8,Operational hubs and control points
1,customers.csv,650,Customer master data with engagement and loyal...
2,drivers.csv,170,Driver workforce data with training and rating...
3,vehicles.csv,120,Fleet asset data including battery and mainten...
4,orders.csv,1250,Service orders across mobility and logistics o...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   file_name     9 non-null      object
 1   record_count  9 non-null      int64 
 2   description   9 non-null      object
dtypes: int64(1), object(2)
memory usage: 348.0+ bytes
None


In [ ]:
data_dictionary = basic_cleaning(data_dictionary)

display(data_dictionary.head())

,file_name,record_count,description
0,hubs.csv,8,Operational hubs and control points
1,customers.csv,650,Customer master data with engagement and loyal...
2,drivers.csv,170,Driver workforce data with training and rating...
3,vehicles.csv,120,Fleet asset data including battery and mainten...
4,orders.csv,1250,Service orders across mobility and logistics o...


In [ ]:
save_cleaned_file(data_dictionary, "data_dictionary_cleaned.csv")

data_dictionary_cleaned.csv saved successfully.


# Checking cleaned CSV files

This step checks that each cleaned CSV file has been saved successfully.

In [ ]:
for file in cleaned_folder.glob("*.csv"):
    print(file.name)

# Create cleaning summary

This table shows the effect of cleaning.

It records:
- raw row count;
- cleaned row count;
- raw column count;
- cleaned column count;
- duplicate rows removed;
- missing values after cleaning.

In [ ]:
cleaned_dataframes = {
    "customers": customers,
    "orders": orders,
    "deliveries": deliveries,
    "drivers": drivers,
    "vehicles": vehicles,
    "hubs": hubs,
    "complaints": complaints,
    "incidents": incidents,
    "app_events": app_events,
    "data_dictionary": data_dictionary
}

raw_dataframes = {
    "customers": customers_raw,
    "orders": orders_raw,
    "deliveries": deliveries_raw,
    "drivers": drivers_raw,
    "vehicles": vehicles_raw,
    "hubs": hubs_raw,
    "complaints": complaints_raw,
    "incidents": incidents_raw,
    "app_events": app_events_raw,
    "data_dictionary": data_dictionary_raw
}

summary_rows = []

for name in cleaned_dataframes:
    raw_df = raw_dataframes[name]
    clean_df = cleaned_dataframes[name]

    summary_rows.append({
        "file_name": name + ".csv",
        "raw_rows": raw_df.shape[0],
        "cleaned_rows": clean_df.shape[0],
        "raw_columns": raw_df.shape[1],
        "cleaned_columns": clean_df.shape[1],
        "duplicates_removed": raw_df.duplicated().sum(),
        "missing_values_after_cleaning": clean_df.isnull().sum().sum()
    })

cleaning_summary = pd.DataFrame(summary_rows)

display(cleaning_summary)

cleaning_summary.to_csv(cleaned_csv_folder / "cleaning_summary.csv", index=False)

,file_name,raw_rows,cleaned_rows,raw_columns,cleaned_columns,duplicates_removed,missing_values_after_cleaning
0,customers.csv,650,650,9,9,0,33
1,orders.csv,1250,1250,11,11,0,25
2,deliveries.csv,950,950,13,14,0,116
3,drivers.csv,170,170,8,8,0,7
4,vehicles.csv,120,120,8,8,0,4
5,hubs.csv,8,8,5,5,0,0
6,complaints.csv,320,320,10,10,0,16
7,incidents.csv,280,280,7,7,0,17
8,app_events.csv,640,640,10,10,0,144
9,data_dictionary.csv,9,9,3,3,0,0


# Creating the CSV-to-JSON function

This function converts cleaned CSV files into JSON format.

This prepares the cleaned data for MongoDB because MongoDB stores data as document-style records.

In [ ]:
def convert_csv_to_json(csv_file, json_folder):
    df = pd.read_csv(csv_file)

    json_file_name = csv_file.stem.replace("_cleaned", "") + ".json"
    json_path = json_folder / json_file_name

    df.to_json(
        json_path,
        orient="records",
        indent=4,
        date_format="iso"
    )

    print(f"Converted {csv_file.name} to {json_file_name}")


cleaned_csv_files = list(cleaned_csv_folder.glob("*_cleaned.csv"))

for csv_file in cleaned_csv_files:
    convert_csv_to_json(csv_file, json_folder)

print("All cleaned CSV files converted to JSON.")

Converted app_events_cleaned.csv to app_events.json
Converted vehicles_cleaned.csv to vehicles.json
Converted deliveries_cleaned.csv to deliveries.json
Converted customers_cleaned.csv to customers.json
Converted hubs_cleaned.csv to hubs.json
Converted incidents_cleaned.csv to incidents.json
Converted orders_cleaned.csv to orders.json
Converted drivers_cleaned.csv to drivers.json
Converted complaints_cleaned.csv to complaints.json
Converted data_dictionary_cleaned.csv to data_dictionary.json
All cleaned CSV files converted to JSON.


# Converting all cleaned CSV files to JSON

This step applies the conversion function to every cleaned CSV file.

The JSON files will be used later for:
- MongoDB Atlas upload;
- PyMongo operations;
- CRUD operations;
- aggregation queries.

In [ ]:
cleaned_csv_files = list(cleaned_folder.glob("*_cleaned.csv"))

for csv_file in cleaned_csv_files:
    convert_csv_to_json(csv_file, cleaned_folder)

print("All cleaned CSV files converted to JSON.")

All cleaned CSV files converted to JSON.


# Checking the JSON files

This step confirms that the JSON files were created successfully.

In [ ]:
for file in cleaned_folder.glob("*.json"):
    print(file.name)

# Checking the cleaned output folders

This step checks that the cleaned CSV and JSON files have been saved into normal folders.

The files are not zipped because they need to be visible individually in GitHub.

In [ ]:
print("Cleaned CSV files:")
for file in cleaned_folder.glob("*.csv"):
    print(file.name)

print("\nJSON files:")
for file in cleaned_folder.glob("*.json"):
    print(file.name)

Cleaned CSV files:

JSON files:


# Loading the clean CSV Files

In [ ]:
cleaned_datasets = {}

for file in cleaned_csv_folder.glob("*_cleaned.csv"):
    dataset_name = file.stem.replace("_cleaned", "")
    cleaned_datasets[dataset_name] = pd.read_csv(file)
    print(f"{dataset_name}: {cleaned_datasets[dataset_name].shape[0]} rows, {cleaned_datasets[dataset_name].shape[1]} columns")

app_events: 640 rows, 10 columns
vehicles: 120 rows, 8 columns
deliveries: 950 rows, 14 columns
customers: 650 rows, 9 columns
hubs: 8 rows, 5 columns
incidents: 280 rows, 7 columns
orders: 1250 rows, 11 columns
drivers: 170 rows, 8 columns
complaints: 320 rows, 10 columns
data_dictionary: 9 rows, 3 columns


In [ ]:
import pandas as pd
from pathlib import Path

cleaned_csv_folder = Path("/content/northstar-databases-analytics/data/cleaned/csv")

customers = pd.read_csv(cleaned_csv_folder / "customers_cleaned.csv")
orders = pd.read_csv(cleaned_csv_folder / "orders_cleaned.csv")
deliveries = pd.read_csv(cleaned_csv_folder / "deliveries_cleaned.csv")
drivers = pd.read_csv(cleaned_csv_folder / "drivers_cleaned.csv")
vehicles = pd.read_csv(cleaned_csv_folder / "vehicles_cleaned.csv")
hubs = pd.read_csv(cleaned_csv_folder / "hubs_cleaned.csv")
complaints = pd.read_csv(cleaned_csv_folder / "complaints_cleaned.csv")
incidents = pd.read_csv(cleaned_csv_folder / "incidents_cleaned.csv")
app_events = pd.read_csv(cleaned_csv_folder / "app_events_cleaned.csv")

print("Cleaned datasets loaded successfully.")

Cleaned datasets loaded successfully.


In [ ]:
from pathlib import Path

repo_folder = Path("/content/northstar-databases-analytics")

cleaned_csv_folder = repo_folder / "data/cleaned/csv"
json_folder = repo_folder / "data/cleaned/json"

print("Cleaned CSV folder:", cleaned_csv_folder)
print("JSON folder:", json_folder)

Cleaned CSV folder: /content/northstar-databases-analytics/data/cleaned/csv
JSON folder: /content/northstar-databases-analytics/data/cleaned/json


In [ ]:
print("Cleaned CSV files:")
for file in cleaned_csv_folder.glob("*.csv"):
    print(file.name)

print("\nJSON files:")
for file in json_folder.glob("*.json"):
    print(file.name)

Cleaned CSV files:
app_events_cleaned.csv
vehicles_cleaned.csv
deliveries_cleaned.csv
customers_cleaned.csv
hubs_cleaned.csv
cleaning_summary.csv
incidents_cleaned.csv
orders_cleaned.csv
drivers_cleaned.csv
complaints_cleaned.csv
data_dictionary_cleaned.csv

JSON files:
hubs.json
drivers.json
app_events.json
deliveries.json
vehicles.json
orders.json
incidents.json
complaints.json
data_dictionary.json
customers.json
